<a href="https://colab.research.google.com/github/mamontoff2018-lgtm/BerezinaAlg/blob/main/practices/LR4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ЛАБОРАТОРНАЯ РАБОТА №4. Наивный байесовский классификатор (Naive Bayes)

## Учебная задача


## Задания


In [1]:
import numpy as np

class NaiveBayes:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # Инициализируем среднее, дисперсию и априорные вероятности
        self.mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self.var = np.zeros((n_classes, n_features), dtype=np.float64)
        self.priors = np.zeros(n_classes, dtype=np.float64)

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.var[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(n_samples)

    def predict(self, X):
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        for idx, c in enumerate(self.classes):
            # Используем логарифмы, чтобы избежать зануления при умножении малых чисел
            prior = np.log(self.priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = posterior + prior
            posteriors.append(posterior)

        return self.classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        # Добавляем небольшое число (1e-9) в знаменатель, чтобы избежать деления на ноль
        numerator = np.exp(- (x - mean)**2 / (2 * var + 1e-9))
        denominator = np.sqrt(2 * np.pi * var + 1e-9)
        return numerator / denominator

In [2]:
if __name__ == "__main__":
    from sklearn.model_selection import train_test_split
    from sklearn.datasets import make_classification

    # Генерируем данные
    X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=123)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

    nb = NaiveBayes()
    nb.fit(X_train, y_train)
    predictions = nb.predict(X_test)

    accuracy = np.sum(predictions == y_test) / len(y_test)
    print(f"Точность модели: {accuracy * 100:.2f}%")

Точность модели: 96.50%


## Контрольные вопросы

1. Почему классификатор называется «Наивным»? К каким последствиям для точности это может привести?

Классификатор называется «наивным», потому что он делает сильное предположение об условной независимости всех признаков для заданного класса — будто каждый признак вносит вклад в вероятность класса независимо от других. В реальных данных признаки часто коррелируют, поэтому данное предположение не выполняется.

Последствия для точности:

Если признаки действительно независимы (или почти независимы), наивный байесовский классификатор работает почти оптимально.

При сильной корреляции признаков оценки вероятностей могут искажаться (вероятность совместного события завышается или занижается), что способно снизить точность. Однако на практике классификатор часто показывает достойные результаты даже при нарушении этого предположения.

2. Что такое априорная вероятность в контексте кода? Как она вычисляется в методе fit?

Априорная вероятность — это безусловная вероятность принадлежности объекта к определённому классу до того, как мы увидели значения его признаков.

Это оценка по закону больших чисел: частота появления объектов класса в обучающей выборке служит приближением истинной априорной вероятности.

3. Какое распределение (кроме Гауссовского) можно использовать для классификации текстовых документов?

Для классификации текстовых документов обычно используют мультиномиальное распределение (Multinomial Naive Bayes). В этой модели признаки представляют собой частоты слов (или n-грамм) в документе. Мультиномиальное распределение естественно моделирует количество появлений каждого слова, что хорошо подходит для анализа текстов.

4. Как изменится работа алгоритма, если один из признаков будет сильно коррелировать с другим? Нарушит ли это работу кода технически?

Технически код не сломается — он выполнит все вычисления без ошибок. Алгоритму всё равно, зависимы признаки или нет: он просто перемножает (суммирует логарифмы) плотности вероятностей.

Однако качество классификации может пострадать: предположение о независимости признаков будет нарушено, и оценки условных вероятностей P(x|y) станут неточными. Например, если два признака сильно коррелируют, их совместный вклад в апостериорную вероятность окажется непропорционально большим (фактически один и тот же сигнал учитывается дважды), что может сместить решение в неверную сторону. На практике наивный байес часто остаётся работоспособным, но сильные корреляции — одна из причин, по которой его точность может уступать более сложным моделям.


